# 00 · Basics of Differentiation (Formula‑First, Astrophysics Context)This notebook teaches differentiation from the ground up with **full formulas, symbol definitions, and astrophysics examples**.  No prior memory of astrophysical equations is assumed — each one is introduced here. All examples are **Python‑first** (no manual rule‑drilling) and **unit‑safe** using `astropy.units`.---## 1. What is Differentiation? (Textbook explanation)Differentiation measures **how fast** one quantity changes with respect to another. If $y=f(x)$, the derivative at $x$ is the limit of an average rate of change over a shrinking interval:$$ f'(x)=\frac{dy}{dx}=\lim_{\Delta x\to 0}\,\frac{f(x+\Delta x)-f(x)}{\Delta x}. $$- **Geometric meaning:** $f'(x)$ is the slope of the tangent line to the curve $y=f(x)$ at $x$.- **Physical meaning:** a derivative is a rate. Example: if $r(t)$ is a position, $dr/dt$ is a velocity; if $v(t)$ is a velocity, $dv/dt$ is an acceleration.- **Units:** the units of a derivative are the units of the numerator divided by the units of the denominator. If $r$ is metres and $t$ is seconds, $dr/dt$ is m·s$^{-1}$.- **Notation:** prime $f'(x)$, Leibniz $\frac{dy}{dx}$, and dot for time $\dot x=\frac{dx}{dt}$ are equivalent ways to express the same idea.**Common pitfalls (and how we avoid them here):**- Mixing units when evaluating derivatives — we always attach units with `astropy.units`.- Confusing discrete differences with derivatives — when using sampled data, we can compute finite differences (`np.gradient`) to approximate derivatives.- Forgetting what variable you’re differentiating with respect to — we’ll always state it explicitly in formulas and code.

## 2. When do astronomers take derivatives?- **Kinematics:** $v=dr/dt$, $a=dv/dt$ for orbits, ejecta, jets.- **Cosmology:** $H=\dot a/a$ gives the expansion rate from scale factor $a(t)$.- **Orbital mechanics:** $dr/d\theta$ in Kepler ellipses describes how orbital radius changes with angle.- **Radiative processes:** $du/dT$ for radiation energy density $u(T)=aT^4$.- **Spatial gradients:** $dT/dr$ inside stars; $d\rho/dr$ in nebulae.

---## Example 1 — Hubble Parameter from the Scale Factor**Formula (definition):**$$ H(t)=\frac{\dot a(t)}{a(t)}. $$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $a(t)$ | Cosmic scale factor | (dimensionless) || $\dot a$ | Time derivative of $a$ | s$^{-1}$ || $H(t)$ | Hubble parameter | s$^{-1}$ (often km·s$^{-1}$·Mpc$^{-1}$) |**Model used for illustration (matter‑dominated toy model):**$$ a(t)=\left(\frac{t}{t_0}\right)^{2/3} \quad\Rightarrow\quad H(t)=\frac{\dot a}{a}=\frac{2}{3t}. $$We derive this with SymPy and evaluate it numerically with time in gigayears.

In [ ]:
import sympy as spfrom astropy import units as ut, t0 = sp.symbols('t t0', positive=True)a = (t/t0)**sp.Rational(2,3)H_expr = sp.simplify(sp.diff(a, t)/a)  # 2/(3t)H_expr

In [ ]:
# Evaluate at t0 = 13.8 Gyr (illustrative present age)t_now = 13.8 * u.GyrH_now = (2/(3*t_now)).to(1/u.s)H_now

**Interpretation.** $H(t)$ decreases like $1/t$ in this toy model — the Universe keeps expanding, but the *rate per unit time* slows with age.

---## Example 2 — Kepler Ellipse: $\,dr/d\theta$**Formula (polar form of a Kepler ellipse):**$$ r(\theta)=\frac{a(1-e^2)}{1+e\cos\theta}. $$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $r(\theta)$ | Distance from the focus at true anomaly $\theta$ | m (often AU) || $a$ | Semi‑major axis | m (often AU) || $e$ | Eccentricity ($0\le e<1$) | (dimensionless) || $\theta$ | True anomaly | rad |We compute $\frac{dr}{d\theta}$ symbolically, evaluate it for a specific orbit, and plot both $r(\theta)$ and $dr/d\theta$.

In [ ]:
import numpy as npimport sympy as spfrom sympy import lambdifyimport matplotlib.pyplot as pltfrom astropy import units as ua_sym, e_sym, th = sp.symbols('a_sym e_sym th', positive=True)r = a_sym*(1 - e_sym**2) / (1 + e_sym*sp.cos(th))dr_dth = sp.simplify(sp.diff(r, th))# numerical evaluationa_val_m = (1 * u.AU).to(u.m).valuee_val = 0.5th_grid = np.linspace(0, 2*np.pi, 721)r_fn = lambdify((a_sym, e_sym, th), r, "numpy")dr_fn = lambdify((a_sym, e_sym, th), dr_dth, "numpy")r_vals = r_fn(a_val_m, e_val, th_grid) * u.mdr_vals = dr_fn(a_val_m, e_val, th_grid) * u.m  # per radplt.figure()plt.plot(th_grid, r_vals.to_value(u.AU))plt.xlabel("θ (rad)")plt.ylabel("r (AU)")plt.title("Kepler ellipse: r(θ)")plt.show()plt.figure()plt.plot(th_grid, dr_vals.to_value(u.m))plt.xlabel("θ (rad)")plt.ylabel("dr/dθ (m per rad)")plt.title("Kepler ellipse: slope dr/dθ")plt.show()r_vals[0].to(u.AU), dr_vals[0].to(u.m)

**Interpretation.** Near periapsis (small $\theta$), $|dr/d\theta|$ is large — the radius changes quickly with angle. Near apoapsis, the slope is smaller.

---## Example 3 — Radiation Energy Density Slope**Formula (radiation constant law):**$$ u(T)=a\,T^4, \quad \frac{du}{dT}=4aT^3. $$**Symbols & units**| Symbol | Meaning | Units ||---|---|---|| $u(T)$ | Radiation energy density | J·m$^{-3}$ || $a$ | Radiation constant ($a=4\sigma/c$) | J·m$^{-3}$·K$^{-4}$ || $T$ | Temperature | K |We evaluate the slope $du/dT$ at two temperatures to see the cubic scaling with $T$.

In [ ]:
import sympy as spfrom astropy import units as u, constants as consta_rad = (4*const.sigma_sb/const.c).to(u.J/(u.m**3*u.K**4))du_2p73 = (4*a_rad*(2.73*u.K)**3).to(u.J/(u.m**3*u.K))du_3000 = (4*a_rad*(3000*u.K)**3).to(u.J/(u.m**3*u.K))du_2p73, du_3000, (du_3000/du_2p73).decompose()

**Interpretation.** Because $du/dT\propto T^3$, hot radiation fields are vastly more sensitive to small changes in $T$ than cold ones.

---## Aside — Numeric derivatives from sampled dataIf you only have samples $(x_i, y_i)$ (e.g., from a file), you can approximate derivatives numerically with finite differences:- Forward/backward difference for edges- Central difference internallyIn NumPy: `np.gradient(y, x)` computes a central‑difference slope that mirrors the calculus idea of $dy/dx$.

In [ ]:
import numpy as npfrom astropy import units as ux = np.linspace(0, 10, 101) * u.sy = (5*(x.to_value(u.s))**2) * u.m   # toy: y = 5 t^2dy_dx = np.gradient(y.to_value(u.m), x.to_value(u.s)) * (u.m/u.s)dy_dx[:5]  # ~ 10 t (m/s) near t=0

---## 3. Practice Puzzles (Exam‑style)**Puzzle A — Orbit slope at a given angle**  **Formula:** $$ r(\theta)=\frac{a(1-e^2)}{1+e\cos\theta}. $$  **Task:** For $a=20$ AU, $e=0.6$, compute $\left.\dfrac{dr}{d\theta}\right|_{\theta=\pi/3}$ in metres per radian. Comment on the sign.**Puzzle B — Radiation slope scaling**  **Formula:** $$ u(T)=aT^4,\quad \frac{du}{dT}=4aT^3. $$  **Task:** Evaluate $\dfrac{du}{dT}$ at $T=10$ K and $T=3000$ K and report the ratio $\left(\dfrac{du}{dT}\right)_{3000}/\left(\dfrac{du}{dT}\right)_{10}$.**Puzzle C — Exponential light‑curve decay**  **Model:** $$ L(t)=L_0 e^{-(t-t_0)/\tau}, \quad \frac{dL}{dt}=-\frac{1}{\tau}L(t). $$  **Given:** $L_0=8\times10^{30}$ W, $t_0=0$, $\tau=30$ days.  **Task:** Compute $\left.\dfrac{dL}{dt}\right|_{t=\tau}$ in W·s$^{-1}$ and state whether the source is brightening or fading.

---## 4. Solutions

In [ ]:
import numpy as npimport sympy as spfrom sympy import lambdifyfrom astropy import units as u, constants as const# Puzzle Aa_sym, e_sym, th = sp.symbols('a_sym e_sym th', positive=True)r = a_sym*(1 - e_sym**2) / (1 + e_sym*sp.cos(th))dr_dth = sp.diff(r, th)a_val = (20 * u.AU).to(u.m).valuee_val = 0.6theta_val = np.pi/3dr_num = lambdify((a_sym, e_sym, th), dr_dth, "numpy")(a_val, e_val, theta_val) * u.m  # per rad# Puzzle Ba_rad = (4*const.sigma_sb/const.c).to(u.J/(u.m**3*u.K**4))du10 = (4*a_rad*(10*u.K)**3).to(u.J/(u.m**3*u.K))du3000 = (4*a_rad*(3000*u.K)**3).to(u.J/(u.m**3*u.K))ratio = (du3000/du10).decompose()# Puzzle Ctau = (30*u.day).to(u.s)L0 = 8e30 * u.Wt = tauL_t = L0 * np.exp(-(t/tau).to_value(u.one))dLdt = -(1/tau) * L_t  # negative -> fadingdr_num, du10, du3000, ratio, L_t.to(u.W), dLdt.to(u.W/u.s)